# SOLARNESIA: Spatial Model & GIS Analytics for National Solar Power Plant (PLTS) Distribution
**Mata Kuliah:** IMT01303501 - Komputer Grafis (3 SKS)  
**Kelompok 6:** Wesley Goeinawan (0706012410011), Stevanus Ivan Santoso (0706012410029), Nicholas Gerwin Mawardji (0706012410015)  
**Program Studi:** Informatika, Universitas Ciputra Surabaya (2026/2027)

---
### Ringkasan Alur Pemrosesan Spasial (AFL 2)
1. **Layer 1 (2D MultiPolygons):** Batas daratan kepulauan dan batas administrasi kabupaten/provinsi (Choropleth rata-rata potensi PVOUT nasional).
2. **Layer 2 (0D Points & Masking):** Titik observasi Global Solar Atlas darat murni (3.802 titik) setelah *Spatial Clip* mengeliminasi 1.160 titik laut.
3. **5 km Hotspot Clustering:** Pengelompokan titik tapak potensial berdekatan (≤ 5 km) menjadi 480 sentra klaster terpadu menggunakan algoritma `DBSCAN(eps=5000)`.
4. **Operasi 1 (Buffer Metrik):** Proyeksi EPSG:3857 dengan koreksi $\cos(\text{lat})$, pembentukan koridor distribusi listrik 25 km `buffer(25000)` sesuai standar teknis PLN Tegangan Menengah 20 kV.
5. **Operasi 2 (Spatial Join & Overlay):** Analisis keterjangkauan terhadap 7.217 titik pusat permukiman kecamatan BPS nasional (2.806 terjangkau / 38,9%) serta ekstraksi wilayah administratif interaktif.
6. **Peta Interaktif (Folium):** Rendering visual layer koroplet provinsi, sentra PLTS 5 km, koridor transmisi 25 km, klaster permukiman BPS, dan popup inspeksi klik per sentra pembangkit.

In [11]:
# Cell 0: Setup & Library Imports
import os  # Mengimpor modul OS bawaan Python untuk manipulasi path direktori dan sistem berkas
import numpy as np  # Mengimpor pustaka NumPy untuk komputasi numerik, trigonometri, dan aljabar matriks
import pandas as pd  # Mengimpor pustaka Pandas untuk manipulasi tabel tabular, DataFrame, dan ekspor CSV
import geopandas as gpd  # Mengimpor GeoPandas untuk pengolahan layer spasial, sistem koordinat CRS, dan operasi GIS
from sklearn.cluster import DBSCAN  # Mengimpor algoritma DBSCAN dari Scikit-Learn untuk klastering spasial kedekatan 5 km
import folium  # Mengimpor pustaka Folium untuk membangun visualisasi peta interaktif berbasis Leaflet.js
from folium.plugins import MarkerCluster, MiniMap, Fullscreen, MeasureControl  # Mengimpor modul plugin Folium untuk klaster marker, peta radar mini, fullscreen, dan pengukur jarak

print(f"GeoPandas: {gpd.__version__} | Pandas: {pd.__version__} | Folium: {folium.__version__}")  # Mencetak versi pustaka utama untuk memvalidasi kompatibilitas lingkungan
os.makedirs("outputs", exist_ok=True)  # Membuat folder 'outputs' secara otomatis di sistem jika belum tersedia
print("Output directory ready.")  # Menampilkan pesan konfirmasi bahwa direktori penyimpanan luaran siap digunakan


GeoPandas: 1.2.0 | Pandas: 3.0.2 | Folium: 0.20.0
Output directory ready.


In [12]:
# Cell 1: Ingestion & GeoDataFrame Construction
DATA_DIR = "data"  # Menetapkan nama konstanta direktori tempat seluruh berkas dataset disimpan

# 1. Load Solar Observation Points (Global Solar Atlas)
df_raw = pd.read_csv(os.path.join(DATA_DIR, "indonesia_solar_data.csv"))  # Membaca data mentah observasi radiasi surya dan temperatur dari berkas CSV
raw_pts = gpd.GeoDataFrame(  # Mengonversi DataFrame tabular pandas menjadi objek spasial GeoDataFrame
    df_raw,  # Menggunakan atribut tabel asal dari variabel df_raw
    geometry=gpd.points_from_xy(df_raw["longitude"], df_raw["latitude"]),  # Mengonstruksi geometri Point Shapely dari koordinat bujur (X) dan lintang (Y)
    crs="EPSG:4326"  # Menetapkan sistem referensi koordinat geografis WGS84 berbasis derajat bola bumi
)

# 2. Load Administrative Boundaries (Provinces & Kabupaten)
gdf_prov = gpd.read_file(os.path.join(DATA_DIR, "indonesia_provinces.geojson"))  # Membaca poligon batas administratif 34 provinsi Indonesia dari GeoJSON
gdf_kab = gpd.read_file(os.path.join(DATA_DIR, "indonesia_kabupaten.geojson"))  # Membaca poligon batas administratif 514 kabupaten/kota Indonesia dari GeoJSON

# 3. Load Administrative Sub-Districts (Kecamatan BPS - 7.217 Titik)
df_kec = pd.read_csv(os.path.join(DATA_DIR, "indonesia_kecamatan.csv"))  # Membaca data titik koordinat pusat kecamatan seluruh Indonesia dari publikasi BPS
gdf_kec = gpd.GeoDataFrame(  # Mengonversi data kecamatan menjadi objek spasial GeoDataFrame
    df_kec,  # Menggunakan data atribut kecamatan dari df_kec
    geometry=gpd.points_from_xy(df_kec["Longitude"], df_kec["Latitude"]),  # Membuat geometri Point dari kolom Longitude dan Latitude
    crs="EPSG:4326"  # Menetapkan sistem koordinat WGS84 pada layer kecamatan
)

assert len(raw_pts) == 4962, f"Expected 4962 points, got {len(raw_pts)}"  # Memverifikasi integritas data titik surya mentah tepat 4.962 titik
assert raw_pts.crs.to_epsg() == 4326  # Memvalidasi CRS data surya berada pada EPSG:4326
assert len(gdf_kec) == 7217, f"Expected 7217 kecamatan, got {len(gdf_kec)}"  # Memvalidasi total titik pusat kecamatan BPS tepat 7.217 titik
print(f"Raw Solar Points loaded: {len(raw_pts)} points | CRS: {raw_pts.crs}")  # Menampilkan status jumlah dan CRS titik surya
print(f"Provinces loaded: {len(gdf_prov)} | Kabupaten loaded: {len(gdf_kab)} | Kecamatan loaded: {len(gdf_kec)}")  # Menampilkan status jumlah batas wilayah yang dimuat

assert gdf_kec.crs.to_epsg() == 4326  # Memvalidasi bahwa layer kecamatan menggunakan proyeksi EPSG:4326
print("Input verification passed.")  # Menampilkan konfirmasi bahwa seluruh verifikasi data masukan berhasil


Raw Solar Points loaded: 4962 points | CRS: EPSG:4326
Provinces loaded: 34 | Kabupaten loaded: 519 | Kecamatan loaded: 7217
Input verification passed.


In [13]:
# Cell 2: Spatial Clip / Land-Masking (Op 0)
# Mengeliminasi titik-titik observasi yang jatuh di perairan laut menggunakan poligon daratan kabupaten
land_sj = gpd.sjoin(raw_pts, gdf_kab[["shapeName", "geometry"]], how="inner", predicate="intersects")  # Melakukan spatial join inner antara titik surya dengan poligon pulau kabupaten
land_pts = raw_pts.loc[land_sj.index.unique()].copy()  # Menyaring baris titik darat yang unik untuk mencegah duplikasi titik di perbatasan daerah

ocean_pts_count = len(raw_pts) - len(land_pts)  # Menghitung selisih jumlah titik untuk mengetahui berapa titik laut yang tereliminasi
print(f"Total Raw Points       : {len(raw_pts)}")  # Mencetak total titik observasi sebelum pemotongan
print(f"Eliminated Ocean Points: {ocean_pts_count} (Dropped via Land-Masking)")  # Mencetak jumlah titik perairan laut yang berhasil dibuang
print(f"Pure Land Points       : {len(land_pts)}")  # Mencetak jumlah titik murni yang berada di atas daratan kepulauan Indonesia
assert len(land_pts) == 3802, f"Expected 3802 points, got {len(land_pts)}"  # Memvalidasi secara matematis bahwa titik daratan murni tepat berjumlah 3.802 titik
assert ocean_pts_count == 1160  # Memvalidasi bahwa tepat 1.160 titik laut telah tereliminasi

# Pembersihan Memori: Hapus variabel mentah yang tidak digunakan lagi untuk melegakan RAM
del df_raw, raw_pts, land_sj  # Menghapus dataframe mentah dari memori RAM komputer agar rendering Folium lancar
print("Spatial Clip verification passed & memory cleaned.")  # Menampilkan pesan konfirmasi bahwa pemotongan spasial sukses dan RAM sudah bersih


Total Raw Points       : 4962
Eliminated Ocean Points: 1160 (Dropped via Land-Masking)
Pure Land Points       : 3802
Spatial Clip verification passed & memory cleaned.


In [14]:
# Cell 3: Site Selection & Kandidat PLTS (Layer 2)
# Ambang batas teknis: PVOUT >= Persentil ke-75 (1.430 kWh/kWp) dan Suhu Rata-rata <= 27,5 °C
p75_pvout = 1430.0  # Menetapkan batas minimal radiasi surya berdasarkan persentil ke-75 nasional (1.430 kWh/kWp/tahun)
max_temp = 27.5  # Menetapkan batas maksimal temperatur lingkungan rata-rata (27,5 °C) untuk menjaga efisiensi fotovoltaik

is_plts = (land_pts["pvout"] >= p75_pvout) & (land_pts["temp"] <= max_temp)  # Membuat masker boolean kondisi kelayakan pembangkit surya
plts_candidates = land_pts[is_plts].copy()  # Menyaring titik darat yang lolos kedua kriteria teknis menjadi kandidat PLTS
plts_candidates["role"] = "PLTS_CANDIDATE"  # Menandai label peran fungsional titik sebagai tapak kandidat pembangkit surya

print(f"Kandidat PLTS Terpilih : {len(plts_candidates)} titik (PVOUT >= {p75_pvout} & Temp <= {max_temp}°C)")  # Menampilkan total titik tapak PLTS yang memenuhi kriteria
assert len(plts_candidates) == 513, f"Expected 513 PLTS candidates, got {len(plts_candidates)}"  # Memvalidasi bahwa jumlah kandidat PLTS tepat 513 titik
print("Site Selection verified.")  # Menampilkan konfirmasi bahwa tahap seleksi tapak berhasil diverifikasi


Kandidat PLTS Terpilih : 513 titik (PVOUT >= 1430.0 & Temp <= 27.5°C)
Site Selection verified.


In [15]:
# Cell 4: 5 km Proximity Hotspot Clustering (DBSCAN)
# Titik-titik tapak PLTS dalam radius <= 5 km dikelompokkan ke dalam satu sentra klaster energi
plts_3857 = plts_candidates.to_crs("EPSG:3857")  # Mentransformasikan koordinat geografis (derajat) kandidat PLTS ke proyeksi metrik EPSG:3857 (meter)
coords_m = np.column_stack([plts_3857.geometry.x, plts_3857.geometry.y])  # Menggabungkan koordinat X dan Y meter menjadi matriks 2D sebagai masukan DBSCAN

db_5k = DBSCAN(eps=5000, min_samples=1, metric="euclidean").fit(coords_m)  # Menjalankan klastering DBSCAN dengan jarak radius 5.000 meter (5 km)
plts_candidates["cluster_id"] = db_5k.labels_  # Menyimpan nomor klaster numerik ke dalam GeoDataFrame geografis kandidat PLTS
plts_3857["cluster_id"] = db_5k.labels_  # Menyimpan nomor klaster numerik ke dalam GeoDataFrame metrik kandidat PLTS

plts_candidates["hotspot_id"] = [f"HOTSPOT-{cid:03d}" for cid in db_5k.labels_]  # Membuat format teks identitas sentra unik (misal: HOTSPOT-000, HOTSPOT-001)
plts_3857["hotspot_id"] = plts_candidates["hotspot_id"]  # Menyinkronkan identitas hotspot string ke layer proyeksi metrik

unique_hotspots = plts_candidates["hotspot_id"].nunique()  # Menghitung jumlah sentra klaster hotspot unik yang terbentuk
multi_site_hotspots = plts_candidates.groupby("hotspot_id").filter(lambda g: len(g) > 1)["hotspot_id"].nunique()  # Menghitung sentra yang menggabungkan lebih dari 1 titik tapak

print(f"Total Kandidat PLTS       : {len(plts_candidates)} titik")  # Menampilkan jumlah titik individual pembangkit
print(f"Total Hotspot Sentra      : {unique_hotspots} klaster")  # Menampilkan total sentra klaster terpadu setelah penggabungan
print(f"Hotspot Multi-titik (<=5km): {multi_site_hotspots} klaster terpadu")  # Menampilkan sentra yang berhasil menyatukan beberapa tapak berdekatan

assert unique_hotspots == 480, f"Expected 480 hotspots, got {unique_hotspots}"  # Memverifikasi bahwa total hotspot sentra tepat 480 klaster
assert multi_site_hotspots == 31  # Memverifikasi bahwa jumlah sentra multi-tapak tepat 31 klaster
print("5km Proximity Clustering verified.")  # Menampilkan konfirmasi keberhasilan pengelompokan kedekatan 5 km


Total Kandidat PLTS       : 513 titik
Total Hotspot Sentra      : 480 klaster
Hotspot Multi-titik (<=5km): 31 klaster terpadu
5km Proximity Clustering verified.


In [16]:
# Cell 5: Transformasi CRS & Pembuatan Buffer Metrik (Op 1)
# Transformasi ke EPSG:3857 dengan koreksi distorsi Mercator: R_adj = R / cos(latitude)
kec_3857 = gdf_kec.to_crs("EPSG:3857")  # Memproyeksikan titik-titik kecamatan dari WGS84 ke sistem metrik EPSG:3857 untuk pengukuran jarak

lats_plts = plts_candidates["latitude"].values  # Mengekstrak array nilai garis lintang (latitude) dari semua titik kandidat PLTS
rad_adj_25k = 25000.0 / np.cos(np.radians(lats_plts))  # Menghitung radius ekuivalen metrik dengan rumus koreksi distorsi Web Mercator R / cos(lat)

# Buffer 25 km (Standar Distribusi 20 kV PLN)
plts_3857["buffer_25km"] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_25k)]  # Membuat geometri poligon lingkaran buffer 25 km untuk setiap sentra PLTS

# Poligon Union Koridor Distribusi (25 km)
union_25km = gpd.GeoSeries(plts_3857["buffer_25km"], crs="EPSG:3857").union_all()  # Menggabungkan seluruh 513 poligon lingkaran buffer menjadi 1 kesatuan poligon multi-geometri koridor nasional
print("Buffer 25 km generated with latitude-scaling compensation.")  # Menampilkan konfirmasi bahwa pembuatan buffer koridor transmisi telah selesai


Buffer 25 km generated with latitude-scaling compensation.


In [17]:
# Cell 6: Analisis Keterjangkauan Permukiman Spasial (Op 2)
# Evaluasi Riil Tingkat Kecamatan BPS (7.217 Titik Pusat Permukiman)
gdf_kec["covered_25km"] = kec_3857.geometry.within(union_25km)  # Melakukan uji spasial apakah geometri titik kecamatan berada di dalam poligon koridor union 25 km

kec_reached_25 = int(gdf_kec["covered_25km"].sum())  # Menghitung total banyaknya kecamatan yang berhasil terjangkau aliran listrik PLTS
kec_unreached_25 = len(gdf_kec) - kec_reached_25  # Menghitung selisih jumlah kecamatan yang belum terjangkau (berstatus off-grid)
kec_pct_25 = (kec_reached_25 / len(gdf_kec)) * 100  # Menghitung rasio persentase cakupan elektrifikasi kecamatan terhadap total kecamatan Indonesia

print(f"Total Kecamatan BPS              : {len(gdf_kec):,} titik")  # Menampilkan total populasi kecamatan seluruh Indonesia
print(f"Kecamatan Terjangkau 25 km       : {kec_reached_25:,} ({kec_pct_25:.1f}%)")  # Menampilkan jumlah dan persentase kecamatan yang terlayani
print(f"Kecamatan Belum Terjangkau 25 km : {kec_unreached_25:,} ({(kec_unreached_25 / len(gdf_kec)) * 100:.1f}%)")  # Menampilkan jumlah dan persentase kecamatan yang masih off-grid

assert kec_reached_25 == 2806, f"Expected 2806, got {kec_reached_25}"  # Memverifikasi secara matematis bahwa jumlah kecamatan terjangkau tepat 2.806 titik
assert kec_unreached_25 == 4411, f"Expected 4411, got {kec_unreached_25}"  # Memverifikasi secara matematis bahwa jumlah kecamatan off-grid tepat 4.411 titik
assert round(kec_pct_25, 1) == 38.9  # Memvalidasi bahwa angka persentase cakupan nasional tepat bernilai 38,9%
print("Kecamatan reach statistics verified.")  # Menampilkan konfirmasi keberhasilan seluruh uji statistik keterjangkauan kecamatan


Total Kecamatan BPS              : 7,217 titik
Kecamatan Terjangkau 25 km       : 2,806 (38.9%)
Kecamatan Belum Terjangkau 25 km : 4,411 (61.1%)
Kecamatan reach statistics verified.


In [18]:
# Cell 7: Ekstraksi Cakupan Wilayah Administratif per Titik / Hotspot
# Melakukan spatial join antara buffer 25 km setiap titik PLTS dengan poligon batas kabupaten
kab_3857 = gdf_kab.to_crs("EPSG:3857")  # Memproyeksikan layer poligon kabupaten ke CRS metrik EPSG:3857 agar selaras dengan buffer metrik

print("Pre-calculating 25km administrative reach for each hotspot...")  # Menampilkan pesan bahwa proses kalkulasi irisan wilayah per hotspot dimulai

covered_districts = []  # Menyiapkan list penampung ringkasan nama daerah jangkauan per sentra PLTS
for idx, geom in enumerate(plts_3857["buffer_25km"]):  # Melakukan iterasi perulangan untuk setiap geometri buffer 25 km
    # Spatial intersection dengan poligon kabupaten
    intersecting_kab = kab_3857[kab_3857.geometry.intersects(geom)]  # Menyeleksi kabupaten yang beririsan spasial secara fisik dengan buffer 25 km
    kab_names = intersecting_kab["shapeName"].tolist()  # Mengekstrak daftar nama kabupaten hasil irisan spasial ke dalam bentuk list Python
    
    if len(kab_names) > 0:  # Memeriksa apakah terdapat minimal satu kabupaten yang beririsan dengan buffer
        if len(kab_names) <= 3:  # Jika jumlah kabupaten yang terjangkau maksimal tiga wilayah
            summary = ", ".join(kab_names)  # Menggabungkan nama-nama kabupaten dipisahkan dengan tanda koma
        else:  # Jika wilayah yang terjangkau melebihi tiga kabupaten
            summary = ", ".join(kab_names[:3]) + f" (+{len(kab_names)-3} wilayah lain)"  # Mengambil 3 nama teratas dan mencantumkan jumlah sisa wilayah lainnya
    else:  # Jika tidak ada irisan administratif darat (misal pulau terpencil atau pesisir)
        summary = "Wilayah Pesisir / Non-Administratif"  # Menetapkan status default untuk pulau kecil atau perairan pesisir
    
    covered_districts.append(summary)  # Menambahkan string ringkasan nama wilayah ke dalam list covered_districts

plts_candidates["covered_areas_25km"] = covered_districts  # Memasukkan data teks wilayah jangkauan ke kolom baru pada GeoDataFrame kandidat PLTS
print("Pre-calculation complete. Sample coverage:")  # Menampilkan status bahwa pra-perhitungan nama wilayah telah selesai
for i in range(min(5, len(plts_candidates))):  # Melakukan perulangan untuk menampilkan 5 sampel pertama sebagai pratinjau hasil
    row = plts_candidates.iloc[i]  # Mengambil baris data sampel ke-i dari tabel plts_candidates
    print(f"  - {row['hotspot_id']} ({row['latitude']:.2f}, {row['longitude']:.2f}): {row['covered_areas_25km']}")  # Mencetak identitas, koordinat, dan daftar kabupaten terjangkau


Pre-calculating 25km administrative reach for each hotspot...
Pre-calculation complete. Sample coverage:
  - HOTSPOT-000 (1.38, 124.62): Kota Manado, Kota Tomohon, Minahasa (+1 wilayah lain)
  - HOTSPOT-001 (-9.99, 120.40): Sumba Timur
  - HOTSPOT-002 (-8.38, 118.18): Bima, Dompu
  - HOTSPOT-003 (-10.34, 123.75): Kota Kupang, Kupang
  - HOTSPOT-004 (0.35, 127.66): Halmahera Selatan, Halmahera Tengah, Kota Tidore Kepulauan


In [19]:
# Cell 8: Rangkuman Metrik Tabel 1 (Analisis Keterjangkauan Permukiman Kecamatan BPS)
summary_table1 = pd.DataFrame([  # Membentuk objek DataFrame untuk merangkum hasil analisis dalam format tabel terstruktur
    {  # Kamus baris data ringkasan parameter
        "Parameter Jaringan": "Radius Buffer 25 km (Standar Saluran Distribusi 20 kV PLN)",  # Menjelaskan spesifikasi parameter jarak transmisi distribusi PLN
        "Total Kecamatan": f"{len(gdf_kec):,}".replace(",", "."),  # Memformat total titik kecamatan nasional dengan pemisah ribuan titik
        "Kecamatan Terjangkau": f"{kec_reached_25:,} ({kec_pct_25:.1f}%)".replace(",", "."),  # Menyajikan jumlah dan persentase kecamatan yang berhasil terjangkau listrik
        "Kecamatan Off-Grid": f"{kec_unreached_25:,} ({(kec_unreached_25 / len(gdf_kec)) * 100:.1f}%)".replace(",", ".")  # Menyajikan jumlah dan persentase kecamatan yang belum terjangkau
    }
])

print("Tabel 1: Hasil Analisis Spatial Join Jangkauan Layanan PLTS:")  # Mencetak teks judul tabel ke layar konsol
print(summary_table1.to_string(index=False))  # Menampilkan tabel ringkasan ke konsol tanpa kolom indeks numerik bawaan pandas

# Ekspor tabel ringkasan ke file CSV
summary_table1.to_csv("outputs/solarnesia_table1_summary.csv", index=False)  # Menyimpan hasil tabel ringkasan ke file CSV di folder outputs
print("Tabel 1 berhasil diekspor ke outputs/solarnesia_table1_summary.csv")  # Mengonfirmasi keberhasilan ekspor file CSV

summary_table1  # Mengembalikan objek DataFrame agar dirender sebagai tabel visual HTML di Jupyter Notebook


Tabel 1: Hasil Analisis Spatial Join Jangkauan Layanan PLTS:
                                        Parameter Jaringan Total Kecamatan Kecamatan Terjangkau Kecamatan Off-Grid
Radius Buffer 25 km (Standar Saluran Distribusi 20 kV PLN)           7.217        2.806 (38.9%)      4.411 (61.1%)
Tabel 1 berhasil diekspor ke outputs/solarnesia_table1_summary.csv


,Parameter Jaringan,Total Kecamatan,Kecamatan Terjangkau,Kecamatan Off-Grid
0,Radius Buffer 25 km (Standar Saluran Distribus...,7.217,2.806 (38.9%),4.411 (61.1%)


In [20]:
# Cell 9: Peta Spasial Interaktif & Inspeksi Klik Terintegrasi (Folium)
# Mengintegrasikan:
# 1. Layer 1: Koroplet Potensi Surya 34 Provinsi (Choropleth Poligon)
# 2. Layer 2: Tapak Sentra PLTS 5 km & Koridor Transmisi 25 km (Buffer Geometris)
# 3. Layer 2: Permukiman Tingkat Kecamatan (7.217 Titik BPS dengan Marker Clustering)
# 4. Fitur Interaktif Tambahan: Measure Control (Pengukur Jarak), MiniMap Radar, Fullscreen, & Floating HUD

import folium  # Mengimpor pustaka Folium untuk membangun visualisasi peta interaktif web berbasis Leaflet.js
from folium.plugins import MarkerCluster, MiniMap, Fullscreen, MeasureControl  # Mengimpor modul plugin Folium untuk klastering, radar minimap, fullscreen, dan pengukur jarak

# Center map on Indonesia (Bali & Java focal area)
center_lat, center_lon = -8.40, 115.20  # Menetapkan koordinat titik tengah awal pandang kamera peta di kawasan Bali dan Jawa Timur
m = folium.Map(  # Menginisialisasi objek kanvas peta interaktif utama Folium
    location=[center_lat, center_lon],  # Mengatur koordinat titik jangkar awal kamera peta berdasarkan lintang dan bujur
    zoom_start=9,  # Menetapkan tingkat pembesaran (zoom level) awal kamera ke skala regional level 9
    tiles="OpenStreetMap",  # Memilih tema peta dasar (basemap tile) standar OpenStreetMap yang informatif
    control_scale=True  # Menampilkan penggaris skala metrik interaktif di sudut kiri bawah peta
)

# -------------------------------------------------------------
# 1. Layer 1: Koroplet Potensi Surya Provinsi (34 Provinsi)
# -------------------------------------------------------------
prov_means = gpd.sjoin(land_pts, gdf_prov, how='inner', predicate='intersects').groupby('state')['pvout'].mean()  # Menghitung rata-rata potensi radiasi surya (PVOUT) per provinsi melalui spatial join
gdf_prov['mean_pvout'] = gdf_prov['state'].map(prov_means).fillna(land_pts['pvout'].mean()).round(1)  # Memetakan nilai rata-rata PVOUT ke poligon provinsi dan mengisi nilai kosong dengan rata-rata nasional

prov_layer = folium.FeatureGroup(name="🗺️ Layer 1: Koroplet Potensi Surya Provinsi (34 Provinsi)", show=False)  # Membuat grup layer koroplet provinsi (status default tidak aktif agar tidak membebani browser)

def prov_style(feature):  # Mendefinisikan fungsi pewarnaan tematik poligon provinsi berdasarkan tingkatan nilai radiasi surya
    val = feature['properties'].get('mean_pvout', 1400)  # Mengambil nilai potensi radiasi surya rata-rata dari atribut fitur GeoJSON provinsi
    if val >= 1500:  # Kondisi jika radiasi surya sangat tinggi (>= 1.500 kWh/kWp)
        c = '#dc2626'  # Menetapkan warna merah tua untuk kelas potensi radiasi sangat tinggi
    elif val >= 1420:  # Kondisi jika radiasi surya tinggi (>= 1.420 kWh/kWp)
        c = '#ea580c'  # Menetapkan warna jingga pekat untuk kelas potensi radiasi tinggi
    elif val >= 1350:  # Kondisi jika radiasi surya sedang (>= 1.350 kWh/kWp)
        c = '#f59e0b'  # Menetapkan warna kuning emas untuk kelas potensi radiasi sedang
    else:  # Kondisi jika radiasi surya di bawah 1.350 kWh/kWp
        c = '#fde047'  # Menetapkan warna kuning cerah untuk kelas potensi radiasi dasar
    return {'fillColor': c, 'color': '#334155', 'weight': 1.2, 'fillOpacity': 0.35}  # Mengembalikan kamus konfigurasi styling batas garis dan transparansi warna poligon

folium.GeoJson(  # Mengonversi layer vektor poligon provinsi menjadi objek GeoJSON visual di peta
    gdf_prov,  # Memasukkan GeoDataFrame poligon 34 provinsi sebagai sumber data spasial
    style_function=prov_style,  # Menerapkan fungsi pewarnaan tematik dinamis prov_style pada setiap poligon provinsi
    tooltip=folium.GeoJsonTooltip(fields=['state', 'mean_pvout'], aliases=['Provinsi:', 'Rata-rata PVOUT (kWh/kWp):'])  # Menampilkan tooltip informasi nama provinsi dan angka radiasi saat kursor melayang
).add_to(prov_layer)  # Memasukkan elemen grafis GeoJSON provinsi ke dalam grup prov_layer
prov_layer.add_to(m)  # Memasukkan grup layer koroplet provinsi ke kanvas peta utama Folium

# -------------------------------------------------------------
# 2. Layer 2: Tapak Sentra PLTS 5 km & Koridor Distribusi 25 km
# -------------------------------------------------------------
# Buffer 5 km: Footprint fisik tapak klaster PLTS
hotspot_5k_layer = folium.FeatureGroup(name="☀️ Tapak Sentra PLTS (Footprint 5 km)", show=True)  # Membuat grup layer tapak fisik sentra klaster PLTS (status default aktif)
for idx, row in plts_candidates.iterrows():  # Melakukan iterasi perulangan untuk setiap baris data sentra pembangkit PLTS
    folium.Circle(  # Menggambar lingkaran berbasis ukuran meter fisik daratan nyata pada peta
        location=[row['latitude'], row['longitude']],  # Menetapkan titik koordinat pusat lingkaran berdasarkan posisi tapak PLTS
        radius=5000,  # Menetapkan radius lingkaran sebesar 5.000 meter (5 km) di permukaan darat
        color="#f59e0b",  # Memberi warna garis batas lingkaran dengan warna jingga matahari
        weight=2,  # Mengatur ketebalan garis batas lingkaran sebesar 2 piksel
        fill=True,  # Mengaktifkan isian warna di dalam lingkaran tapak pembangkit
        fill_color="#f59e0b",  # Menetapkan warna isi lingkaran dengan warna jingga cerah
        fill_opacity=0.35,  # Mengatur transparansi warna isi sebesar 35% agar peta dasar tetap terlihat jelas
        tooltip=f"<b>Hotspot:</b> {row['hotspot_id']}<br><b>Footprint Sentra:</b> 5 km"  # Menampilkan tooltip identitas sentra saat kursor melayang di atas lingkaran
    ).add_to(hotspot_5k_layer)  # Memasukkan lingkaran tapak 5 km ke dalam grup layer hotspot_5k_layer
hotspot_5k_layer.add_to(m)  # Memasukkan layer tapak 5 km ke kanvas peta utama

# Buffer 25 km: Batas teknis keandalan transmisi TM 20 kV PLN
buffer_25k_layer = folium.FeatureGroup(name="⚡ Koridor Distribusi 25 km (PLN 20 kV)", show=True)  # Membuat grup layer koridor transmisi tegangan menengah 25 km (status default aktif)
for idx, row in plts_candidates.iterrows():  # Melakukan iterasi perulangan untuk setiap baris data sentra pembangkit PLTS
    folium.Circle(  # Menggambar lingkaran berbasis ukuran meter fisik daratan nyata pada peta
        location=[row['latitude'], row['longitude']],  # Menetapkan titik koordinat pusat lingkaran berdasarkan posisi tapak PLTS
        radius=25000,  # Menetapkan radius koridor transmisi sebesar 25.000 meter (25 km) sesuai standar PLN 20 kV
        color="#38bdf8",  # Memberi warna garis batas lingkaran dengan warna biru muda elektrik
        weight=1.5,  # Mengatur ketebalan garis batas koridor sebesar 1,5 piksel
        fill=True,  # Mengaktifkan isian warna di dalam lingkaran tapak pembangkit
        fill_color="#38bdf8",  # Menetapkan warna isi koridor dengan warna biru langit
        fill_opacity=0.12,  # Mengatur transparansi warna isi sebesar 12% agar tidak menghalangi detail layer lain
        tooltip=f"<b>Hotspot:</b> {row['hotspot_id']}<br><b>Radius Distribusi:</b> 25 km"  # Menampilkan tooltip jangkauan distribusi saat kursor berada di atas koridor
    ).add_to(buffer_25k_layer)  # Memasukkan lingkaran koridor 25 km ke dalam grup buffer_25k_layer
buffer_25k_layer.add_to(m)  # Memasukkan layer koridor transmisi 25 km ke kanvas peta utama

# -------------------------------------------------------------
# 3. Layer 2: Permukiman Berbasis Kecamatan (7.217 Titik BPS)
# -------------------------------------------------------------
# Menggunakan gdf_kec hasil komputasi end-to-end (Cell 1 & Cell 6)
kec_gdf = gdf_kec  # Membuat alias variabel layer kecamatan untuk mempermudah pemanggilan kode

# A. Kecamatan Terjangkau 25 km (Emerald Cluster)
kec_reached_cluster = MarkerCluster(name="🟢 Permukiman Kecamatan Terjangkau (2.806 BPS)", show=True)  # Membuat klaster marker untuk titik kecamatan yang terlayani listrik (default aktif)
for idx, row in kec_gdf[kec_gdf['covered_25km']].iterrows():  # Melakukan iterasi hanya pada kecamatan yang berstatus terjangkau (covered_25km == True)
    folium.CircleMarker(  # Menggambar marker lingkaran berbasis ukuran piksel layar tetap (tidak membesar saat di-zoom)
        location=[row['Latitude'], row['Longitude']],  # Menetapkan koordinat posisi titik pusat kecamatan
        radius=4,  # Menetapkan ukuran jari-jari penanda tetap sebesar 4 piksel layar monitor
        color="#059669",  # Memberi warna garis batas penanda dengan hijau zamrud pekat
        weight=1.5,  # Mengatur ketebalan garis batas koridor sebesar 1,5 piksel
        fill=True,  # Mengaktifkan isian warna di dalam lingkaran tapak pembangkit
        fill_color="#10b981",  # Memberi warna hijau terang untuk melambangkan kecamatan berlistrik energi hijau
        fill_opacity=0.9,  # Mengatur tingkat kepekatan warna isi sebesar 90% agar terlihat jelas
        tooltip=f"<b>🏛️ Kec. {row['Name']}</b> ({row['Kabupaten']})<br><span style='color:#059669; font-weight:bold;'>⚡ Terjangkau Feeder 25 km</span>"  # Tooltip nama kecamatan, kabupaten, dan status jangkauan listrik
    ).add_to(kec_reached_cluster)  # Memasukkan marker kecamatan terjangkau ke dalam grup klaster kec_reached_cluster
kec_reached_cluster.add_to(m)  # Memasukkan klaster penanda kecamatan terjangkau ke kanvas peta utama

# B. Kecamatan Belum Terjangkau / Off-Grid (Slate Cluster - Toggleable)
kec_unreached_cluster = MarkerCluster(name="⚪ Permukiman Kecamatan Off-Grid (4.411 BPS)", show=False)  # Membuat klaster marker untuk kecamatan di luar jangkauan (default non-aktif agar ringan)
for idx, row in kec_gdf[~kec_gdf['covered_25km']].iterrows():  # Melakukan iterasi hanya pada kecamatan yang belum terjangkau (~covered_25km)
    folium.CircleMarker(  # Menggambar marker lingkaran berbasis ukuran piksel layar tetap (tidak membesar saat di-zoom)
        location=[row['Latitude'], row['Longitude']],  # Menetapkan koordinat posisi titik pusat kecamatan
        radius=3,  # Menetapkan ukuran jari-jari penanda tetap sebesar 3 piksel layar monitor
        color="#64748b",  # Memberi warna garis batas penanda abu-abu netral
        weight=1,  # Mengatur ketebalan garis batas penanda sebesar 1 piksel
        fill=True,  # Mengaktifkan isian warna di dalam lingkaran tapak pembangkit
        fill_color="#94a3b8",  # Memberi warna abu-abu terang untuk menandakan daerah terisolir/off-grid
        fill_opacity=0.6,  # Mengatur transparansi warna penanda sebesar 60%
        tooltip=f"<b>🏛️ Kec. {row['Name']}</b> ({row['Kabupaten']})<br><span style='color:#dc2626;'>⚪ Di luar Jangkauan 25 km (Off-Grid)</span>"  # Tooltip nama wilayah dan status off-grid
    ).add_to(kec_unreached_cluster)  # Memasukkan marker kecamatan off-grid ke dalam grup klaster kec_unreached_cluster
kec_unreached_cluster.add_to(m)  # Memasukkan klaster penanda kecamatan off-grid ke kanvas peta utama

# -------------------------------------------------------------
# 4. Layer 2: Kandidat PLTS dengan Popup Inspeksi Klik
# -------------------------------------------------------------
plts_layer = folium.FeatureGroup(name="☀️ Titik Kandidat PLTS (513 Titik)", show=True)  # Membuat grup layer titik lokasi sentra pembangkit surya (status default aktif)
for idx, row in plts_candidates.iterrows():  # Melakukan iterasi perulangan untuk setiap baris data sentra pembangkit PLTS
    popup_html = f"""
    <div style='font-family: Arial, sans-serif; font-size: 12px; width: 270px; line-height: 1.5;'>
        <b style='color: #d97706; font-size: 13px;'>☀️ {row['hotspot_id']}</b><br>
        <b>Koordinat:</b> {row['latitude']:.4f}° LS, {row['longitude']:.4f}° BT<br>
        <b>Potensi PVOUT:</b> <span style='color: #059669; font-weight: bold;'>{row['pvout']:.1f} kWh/kWp</span><br>
        <b>Suhu Rata-rata:</b> {row['temp']:.1f} °C<br>
        <hr style='margin: 6px 0; border: 0; border-top: 1px solid #e5e7eb;'>
        <b>Wilayah Terjangkau (Radius 25 km):</b><br>
        <span style='color: #1e3a8a; font-size: 11px;'>{row['covered_areas_25km']}</span>
    </div>
    """
    folium.CircleMarker(  # Menggambar marker lingkaran berbasis ukuran piksel layar tetap (tidak membesar saat di-zoom)
        location=[row['latitude'], row['longitude']],  # Menetapkan titik koordinat pusat lingkaran berdasarkan posisi tapak PLTS
        radius=6.5,  # Mengatur ukuran penanda sebesar 6,5 piksel layar agar mudah dilihat dan diklik oleh pengguna
        color="#ffffff",  # Memberi garis tepi putih kontras setebal 2 piksel agar penanda menonjol di atas peta
        weight=2,  # Mengatur ketebalan garis batas lingkaran sebesar 2 piksel
        fill=True,  # Mengaktifkan isian warna di dalam lingkaran tapak pembangkit
        fill_color="#f59e0b",  # Menetapkan warna isi lingkaran dengan warna jingga cerah
        fill_opacity=0.95,  # Mengatur kepekatan warna isi sebesar 95%
        popup=folium.Popup(popup_html, max_width=300),  # Mengaitkan jendela popup HTML interaktif yang akan terbuka saat marker diklik pengguna
        tooltip=f"<b>Klik untuk Detail:</b> {row['hotspot_id']} (PVOUT: {row['pvout']:.0f})"  # Menampilkan pesan bantuan klik saat kursor melayang di atas titik
    ).add_to(plts_layer)  # Memasukkan penanda titik PLTS ke dalam plts_layer
plts_layer.add_to(m)  # Memasukkan layer kandidat PLTS ke kanvas peta utama

# -------------------------------------------------------------
# 5. Floating Glassmorphism HUD Widget
# -------------------------------------------------------------
hud_html = """
<div style="
    position: fixed;
    bottom: 25px;
    left: 25px;
    z-index: 1000;
    background: rgba(15, 23, 42, 0.90);
    backdrop-filter: blur(8px);
    border: 1px solid rgba(56, 189, 248, 0.35);
    border-radius: 12px;
    padding: 12px 18px;
    color: #f8fafc;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 12px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.45);
    max-width: 300px;
    line-height: 1.5;
">
    <div style="display: flex; align-items: center; justify-content: space-between;">
        <b style="color: #38bdf8; font-size: 13px;">☀️ SOLARNESIA GIS HUD</b>
        <span style="font-size: 10px; background: #0369a1; padding: 2px 6px; border-radius: 4px; color: white;">LIVE</span>
    </div>
    <hr style="margin: 7px 0; border: 0; border-top: 1px solid rgba(255,255,255,0.15);">
    <div>⚡ <b>Kandidat PLTS:</b> 513 Titik (480 Hotspot)</div>
    <div>🟡 <b>Footprint Sentra:</b> 5 km</div>
    <div>🔵 <b>Batas Feeder:</b> 25 km (PLN 20 kV)</div>
    <div style="margin-top: 5px; color: #34d399;">🟢 <b>Kecamatan Terjangkau:</b> 2.806 (38,9%)</div>
    <div style="color: #94a3b8;">⚪ <b>Kecamatan Off-Grid:</b> 4.411 (61,1%)</div>
    <div style="margin-top: 6px; font-size: 10.5px; color: #94a3b8; font-style: italic;">
        💡 Gunakan menu layer kanan atas untuk eksplorasi.
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(hud_html))  # Menyematkan komponen antarmuka widget HUD melayang langsung ke elemen root HTML peta

# -------------------------------------------------------------
# 6. Kontrol Navigasi & Alat GIS Interaktif
# -------------------------------------------------------------
Fullscreen(position='topleft').add_to(m)  # Menambahkan tombol pintas mode layar penuh di sudut kiri atas peta
MeasureControl(position='topleft', primary_length_unit='kilometers').add_to(m)  # Menambahkan alat interaktif pengukur jarak langsung dalam satuan kilometer
MiniMap(toggle_display=True, position='bottomright').add_to(m)  # Menambahkan peta radar kecil yang dapat disembunyikan di sudut kanan bawah
folium.LayerControl(collapsed=False).add_to(m)  # Menambahkan panel kontrol pemilih layer yang terbuka secara default di sudut kanan atas
m.save("outputs/solarnesia_interactive_map.html")  # Menyimpan seluruh peta interaktif beserta data dan styling-nya ke berkas HTML mandiri di folder outputs
print("Interactive Folium map saved to outputs/solarnesia_interactive_map.html.")  # Menampilkan pesan konfirmasi keberhasilan ekspor berkas peta HTML
m  # Menampilkan objek peta interaktif secara visual langsung di sel Jupyter Notebook


Interactive Folium map saved to outputs/solarnesia_interactive_map.html.
